In [1]:
import pandas as pd

# 1. Assessing How Much Data is missing on Observed Job Exposure Data (March 2026) and Anthropic Economic Index Data (August 2026)

The goal of this analysis is to determine how many MCA occupations would be lost due to missing Observed Exposure and Automation and Augmentation values. Of the 1,151 occupations, 1,060 (92%) have Observed Exposure data, 907 (79%) have Anthropic Economic Index data, and 852 (74%) have both.

Since the missing values are determined by whether the SOC code is represented in each dataset, mapping in either direction results in the same coverage constraint. Although the drop-off from Anthropic Economic Index data to occupations with both measures is relatively small (79% to 74%), recalculating job exposure using the newer Anthropic data is preferable since it provides more up-to-date measures while allowing us to retain more occupations rather than relying on requiring a job to have both the Observed Exposure data and Anthropic Economic Index data. 

In [2]:
# have this list just to see the funnel of how many codes have some value for a metric
num_jobs_after_merge = []

In [3]:
# load up first the data of mca and get its set of SOC codes
mca_df = pd.read_csv('../data/auxiliary/final_mca_soc_code.csv')
mca_codes = set(mca_df['SOC Code'])
num_jobs_after_merge.append(mca_df.shape[0])

# get the data from last march that has the observed exposure
occupations_observed_exposure_df = pd.read_csv('../data/ai_measurements/job_exposure.csv')
occupations_observed_exposure_df['Code'] = occupations_observed_exposure_df['occ_code'].astype(str) + '.00'
occupations_observed_exposure_codes = set(occupations_observed_exposure_df['Code'])
codes_obs_exposure_dict = dict(zip(occupations_observed_exposure_df['Code'], occupations_observed_exposure_df['observed_exposure']))

In [4]:
# map SOC codes to observed exposure
mca_df["Observed Exposure"] = mca_df["SOC Code"].map(codes_obs_exposure_dict)

num_mca_codes = len(mca_df)
num_observed_exposure = mca_df["Observed Exposure"].notna().sum()

print(
    f"Out of the {num_mca_codes} jobs in the MCA, "
    f"{num_observed_exposure} ({num_observed_exposure / num_mca_codes:.0%}) "
    "have a corresponding Observed Exposure value in the March 2026 data."
)

num_jobs_after_merge.append(int(num_observed_exposure))

Out of the 1151 jobs in the MCA, 1060 (92%) have a corresponding Observed Exposure value in the March 2026 data.


In [5]:
# get the anthropic data that has automation v augmentation
anthropic_df = pd.read_csv('../data/ai_measurements/release_2026_06_26/data/aei_claude_ai_2026-06-26.csv')
query = (
    "geo_id == 'GLOBAL' and "
    "category_name == 'soc_occupation' and "
    "hierarchy_level == 0 and "
    "date_end == '2026-05-01'"
)
anthropic_global_df = anthropic_df.query(query)
soc_values_df = (
    anthropic_global_df
    .pivot(
        index=['node_name', 'node_external_id'], 
        columns='metric_id', 
        values='value'
    ).reset_index()
)

In [6]:
# merge Anthropic data with MCA
mca_df = mca_df.merge(
    soc_values_df,
    how="left",
    left_on="SOC Code",
    right_on="node_external_id",
)


# count MCA codes with Anthropic data
anthropic_cols = soc_values_df.columns.difference(
    ["node_name", "node_external_id"]
)

has_anthropic_data = mca_df[anthropic_cols].notna().all(axis=1)
num_anthropic_codes = has_anthropic_data.sum()

print(
    f"Out of the {num_mca_codes} jobs in the MCA, "
    f"{num_anthropic_codes} ({num_anthropic_codes / num_mca_codes:.0%}) "
    "have corresponding data in the June 2026 Anthropic Economic Index."
)

Out of the 1151 jobs in the MCA, 907 (79%) have corresponding data in the June 2026 Anthropic Economic Index.


In [7]:
# count MCA codes with both observed exposure and Anthropic data
has_observed_exposure = mca_df["Observed Exposure"].notna()
has_both = has_observed_exposure & has_anthropic_data

num_codes_after_merge = has_both.sum()

print(
    f"Out of the {num_mca_codes} jobs in the MCA, "
    f"{num_codes_after_merge} ({num_codes_after_merge / num_mca_codes:.0%}) "
    "have both Observed Exposure and Anthropic Economic Index data."
)

num_jobs_after_merge.append(int(num_codes_after_merge))

Out of the 1151 jobs in the MCA, 852 (74%) have both Observed Exposure and Anthropic Economic Index data.


# 2. Recalculate the Observed Exposure

In [34]:
# get task level observed exposure and anthropic values
task_observed_exposure = pd.read_csv('../data/ai_measurements/task_penetration.csv')
anthropic_df = pd.read_csv('../data/ai_measurements/release_2026_06_26/data/aei_claude_ai_2026-06-26.csv')
query = (
    "geo_id == 'GLOBAL' and "
    "category_name == 'onet' and " # to get specifically of tasks
    "hierarchy_level == 0 and "
    "date_end == '2026-05-01'"
)
anthropic_tasks_df = anthropic_df.query(query)
task_values_df = (
    anthropic_tasks_df
    .pivot(
        index=['node_name', 'node_external_id'], 
        columns='metric_id', 
        values='value'
    )
)

In [59]:
task_duration_df = task_values_df[['human_only_time_mean']].reset_index()
task_duration_df['node_name'] = task_duration_df['node_name'].str.strip().str.lower()

# 3. Consolidate the data to have AIOE, Observed Exposure, Augmentation versus Automation Percentage